# Mi proyecto · De la teoría a la práctica (Sesiones 1 a 3)
**Plantilla espejo del caso guiado Olist** · Módulo: Preparación, Recopilación y Análisis Exploratorio de Datos · MIADAS – UMSA

**Grupo 4:** Alfredo Martín Velasquez Calizaya, Ana Karen Gutierrez Hurtado, Corina Colque Flores, Gonzalo Mayta Mamani,Gervacio Delgado Ticona
**Conjunto de datos:** Flight Delay Dataset — 2024 · **Fuente y URL:** https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024
**Conjunto de datos:** _Flight Delay Dataset — 2024_ · **Fuente y URL:** _https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024_

Esta plantilla tiene **las mismas secciones** que el caso guiado. Repite cada paso con **tu conjunto de datos**. Este notebook evolucionará en las próximas sesiones con la limpieza, el análisis exploratorio, la normalización y la calidad.

### Requisitos mínimos de tu conjunto de datos
- [x] Al menos **1 000 registros**
- [x] Al menos **8 variables** que mezclen escalas (nominal, ordinal, de razón)
- [x] Al menos **una variable de fecha**
- [x] **Problemas de calidad** reales (faltantes, duplicados o inconsistencias)
- [x] Idealmente, **dos tablas o fuentes** que se puedan integrar
- [x] **Licencia** que permita su uso académico

## Paso 0 · Cargar los datos
Reemplaza la dirección por la URL de tu archivo. Si el archivo está en tu computadora, súbelo con el ícono 📁 de Colab y escribe solo su nombre.

- CSV: `pd.read_csv(url)` · si usa punto y coma: `pd.read_csv(url, sep=";", decimal=",")`
- Excel: `pd.read_excel(url)`
- JSON: `pd.read_json(url)`

In [2]:
import math
import pandas as pd

# 1. Leer el archivo completo que subiste a Colab
df_sample = pd.read_csv("flight_data_2024_sample.csv")

# 2. Crear la clave única flight_id (del 1 al 10,000)
df_sample.insert(0, "flight_id", range(1, len(df_sample) + 1))

# 3. Dividir y guardar las 3 tablas exactas del proyecto
cols_schedule = [
    "flight_id", "year", "month", "day_of_month", "day_of_week", "fl_date",
    "op_unique_carrier", "op_carrier_fl_num", "origin", "origin_city_name",
    "origin_state_nm", "dest", "dest_city_name", "dest_state_nm",
    "crs_dep_time", "crs_arr_time", "crs_elapsed_time", "distance"
]
cols_performance = [
    "flight_id", "dep_time", "dep_delay", "taxi_out", "wheels_off",
    "wheels_on", "taxi_in", "arr_time", "arr_delay"
]
cols_delays = [
    "flight_id", "cancelled", "cancellation_code", "diverted",
    "carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"
]

df_sample[cols_schedule].to_csv("flight_data_2024_sample_flight_schedule.csv", index=False)
df_sample[cols_performance].to_csv("flight_data_2024_sample_flight_performance.csv", index=False)
df_sample[cols_delays].to_csv("flight_data_2024_sample_flight_delays.csv", index=False)

# 4. Cargar la tabla principal en 'datos'
URL = "flight_data_2024_sample_flight_schedule.csv"
datos = pd.read_csv(URL)

print(datos.shape)
datos.head()

(10000, 18)


,flight_id,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,dest,dest_city_name,dest_state_nm,crs_dep_time,crs_arr_time,crs_elapsed_time,distance
0,1,2024,4,18,4,2024-04-18,MQ,3535.0,DFW,"Dallas/Fort Worth, TX",Texas,RAP,"Rapid City, SD",South Dakota,1018,1149,151.0,835.0
1,2,2024,1,1,1,2024-01-01,AA,148.0,CLT,"Charlotte, NC",North Carolina,PHX,"Phoenix, AZ",Arizona,1637,1923,286.0,1773.0
2,3,2024,12,12,4,2024-12-12,9E,5440.0,CHA,"Chattanooga, TN",Tennessee,ATL,"Atlanta, GA",Georgia,1000,1059,59.0,106.0
3,4,2024,4,8,1,2024-04-08,WN,1971.0,OMA,"Omaha, NE",Nebraska,LAS,"Las Vegas, NV",Nevada,1330,1430,180.0,1099.0
4,5,2024,2,16,5,2024-02-16,WN,862.0,BWI,"Baltimore, MD",Maryland,MYR,"Myrtle Beach, SC",South Carolina,1340,1510,90.0,399.0


### Paso 0 (opcional) · Si tu conjunto de datos está en Kaggle
Kaggle no permite leer el archivo por su dirección. Usa la librería oficial **kagglehub**: descarga los datasets **públicos** sin iniciar sesión.

1. Copia el identificador de la dirección del dataset: en `kaggle.com/datasets/olistbr/brazilian-ecommerce` es `olistbr/brazilian-ecommerce`.
2. Quita el `#` de las líneas de la celda, reemplaza el identificador y el nombre del archivo, y ejecuta.

> Si el dataset pide **aceptar reglas** (por ejemplo, los de competencias), necesitarás un token de Kaggle: mejor elige otro dataset.

In [3]:
# !pip install kagglehub        # solo si Colab indica que no está instalada
# import os, kagglehub
# ruta = kagglehub.dataset_download("usuario/nombre-del-dataset")
# print(os.listdir(ruta))                                  # muestra los archivos descargados
# datos = pd.read_csv(os.path.join(ruta, "nombre_del_archivo.csv"))
# print(datos.shape)

---
# S1 · Tipos de datos y escalas (Sesión 1)
### S1.1 · Tipos de dato
¿Qué columnas se leyeron con un tipo incorrecto (fechas como texto, códigos como números)?

In [4]:
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   flight_id          10000 non-null  int64  
 1   year               10000 non-null  int64  
 2   month              10000 non-null  int64  
 3   day_of_month       10000 non-null  int64  
 4   day_of_week        10000 non-null  int64  
 5   fl_date            10000 non-null  object 
 6   op_unique_carrier  10000 non-null  object 
 7   op_carrier_fl_num  10000 non-null  float64
 8   origin             10000 non-null  object 
 9   origin_city_name   10000 non-null  object 
 10  origin_state_nm    10000 non-null  object 
 11  dest               10000 non-null  object 
 12  dest_city_name     10000 non-null  object 
 13  dest_state_nm      10000 non-null  object 
 14  crs_dep_time       10000 non-null  int64  
 15  crs_arr_time       10000 non-null  int64  
 16  crs_elapsed_time   1000

### S1.2 · Corrige los tipos
Convierte las fechas con `pd.to_datetime` y los códigos (identificadores, códigos postales) a texto con `astype(str)`.

In [5]:
datos["fl_date"] = pd.to_datetime(datos["fl_date"])
datos["flight_id"] = datos["flight_id"].astype(str)
datos["op_unique_carrier"] = datos["op_unique_carrier"].astype(str)
datos["op_carrier_fl_num"] = datos["op_carrier_fl_num"].astype(str)
datos["origin"] = datos["origin"].astype(str)
datos["origin_city_name"] = datos["origin_city_name"].astype(str)
datos["origin_state_nm"] = datos["origin_state_nm"].astype(str)
datos["dest"] = datos["dest"].astype(str)
datos["dest_city_name"] = datos["dest_city_name"].astype(str)
datos["dest_state_nm"] = datos["dest_state_nm"].astype(str)

### S1.3 · Escala de cada variable
Completa la tabla con tus variables (mínimo 8):

| Variable | Escala (nominal / ordinal / intervalo / razón) | Cálculos válidos |
|---|---|---|
| `flight_id` | Nominal | Frecuencias, conteo |
| `year` | Intervalo | Diferencias entre años |
| `month` | Ordinal | Frecuencias, porcentajes, moda, mediana |
| `day_of_month` | Intervalo | Diferencias entre días |
| `day_of_week` | Nominal | Frecuencias, moda |
| `fl_date` | Intervalo | Diferencias entre fechas |
| `op_unique_carrier` | Nominal | Frecuencias, porcentajes, moda |
| `op_carrier_fl_num` | Nominal | Frecuencias, moda |
| `origin` | Nominal | Frecuencias, porcentajes, moda |
| `dest` | Nominal | Frecuencias, porcentajes, moda |
| `crs_dep_time` | Intervalo | Diferencias entre horarios |
| `crs_arr_time` | Intervalo | Diferencias entre horarios |
| `crs_elapsed_time` | Razón | Todos: media, desviación estándar, proporciones |
| `distance` | Razón | Todos: media, desviación estándar, proporciones |

In [6]:
# Frecuencias de una variable nominal u ordinal:
datos["flight_id"].value_counts()


,count
flight_id,
1,1
2,1
3,1
4,1
5,1
...,...
9996,1
9997,1
9998,1


In [7]:
# Estadísticos de una variable de razón:
datos[["crs_elapsed_time", "distance"]].describe().round(2)

,crs_elapsed_time,distance
count,10000.00,10000.00
mean,147.37,838.35
std,72.93,598.53
min,23.00,31.00
25%,93.00,404.00
50%,130.00,680.00
75%,179.00,1072.00
max,685.00,5095.00


---
# S2 · Muestreo (Sesión 2)
¿Qué subconjunto necesitarías revisar a mano o encuestar? Calcula el tamaño de muestra y compara una muestra aleatoria con una estratificada.

In [8]:
def tamano_muestra(N, e=0.05, z=1.96, p=0.5):
    n0 = z**2 * p * (1 - p) / e**2
    return math.ceil(n0 / (1 + (n0 - 1) / N))

n = tamano_muestra(len(datos))
n

370

In [9]:
variable_estrato = "op_unique_carrier"
aleatoria = datos.sample(n=n, random_state=42)
estratificada = datos.groupby(variable_estrato, group_keys=False).sample(frac=n / len(datos), random_state=42)
pd.DataFrame({
    "Población": datos[variable_estrato].value_counts(normalize=True),
    "Aleatoria": aleatoria[variable_estrato].value_counts(normalize=True),
    "Estratificada": estratificada[variable_estrato].value_counts(normalize=True),
}).round(3)

,Población,Aleatoria,Estratificada
op_unique_carrier,,,
9E,0.028,0.024,0.027
AA,0.138,0.124,0.138
AS,0.032,0.038,0.033
B6,0.033,0.043,0.033
DL,0.146,0.135,0.146
F9,0.030,0.035,0.030
G4,0.017,0.008,0.016
HA,0.010,0.011,0.011
MQ,0.041,0.054,0.041


---
# S3 · Fuentes e integración (Sesión 3)
Si tu conjunto tiene dos o más tablas, únelas por su clave. Si tiene una sola, identifica qué fuente externa podría complementarla (por ejemplo, datos del INE).

In [10]:
performance = pd.read_csv("flight_data_2024_sample_flight_performance.csv")
performance["flight_id"] = performance["flight_id"].astype(str)
datos_performance = datos.merge(performance, on="flight_id", how="left")

delay = pd.read_csv("flight_data_2024_sample_flight_delays.csv")
delay["flight_id"] = delay["flight_id"].astype(str)
datos_completos = datos_performance.merge(delay, on="flight_id", how="left")

print("Dimensiones finales integradas:", datos_completos.shape)
datos_completos.head(3)

Dimensiones finales integradas: (10000, 34)


,flight_id,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,...,arr_time,arr_delay,cancelled,cancellation_code,diverted,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,1,2024,4,18,4,2024-04-18,MQ,3535.0,DFW,"Dallas/Fort Worth, TX",...,1139.0,-10.0,0,NaN,0,0,0,0,0,0
1,2,2024,1,1,1,2024-01-01,AA,148.0,CLT,"Charlotte, NC",...,1906.0,-17.0,0,NaN,0,0,0,0,0,0
2,3,2024,12,12,4,2024-12-12,9E,5440.0,CHA,"Chattanooga, TN",...,1042.0,-17.0,0,NaN,0,0,0,0,0,0


---
# Cierre · Informe de recolección inicial (CRISP-DM, fase 2)

| Elemento | Mi conjunto de datos |
|---|---|
| **Fuente** | Kaggle – *Flight Data 2024* (autor: Hrishit Patil, basado en registros del Bureau of Transportation Statistics – BTS de EE. UU.): https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024 |
| **Tipo de fuente** | Externa · datos secundarios · registros operacionales de aerolíneas |
| **Formato y acceso** | CSV (3 tablas relacionadas) · descarga desde Kaggle y lectura en Colab |
| **Tablas y número de registros** | Muestra de 10 000 vuelos dividida en 3 tablas relacionales por `flight_id`: `flight_schedule` (10 000 × 18), `flight_performance` (10 000 × 8) y `flight_delays` (10 000 × 8) |
| **Periodo que cubre** | 1 de enero de 2024 al 31 de diciembre de 2024 |
| **Problemas encontrados** | Fecha (`fl_date`) leída como texto (`object`); identificadores (`flight_id`, `op_carrier_fl_num`) leídos como enteros; horas programadas (`crs_dep_time`, `crs_arr_time`) en formato numérico HHMM sin separador; valores nulos estructurales en tiempos reales y demoras correspondientes a vuelos cancelados o desviados |
| **Primer hallazgo o pregunta de negocio** | ¿Qué aerolíneas, rutas y causas operacionales (clima, aerolínea, tráfico aéreo o aeronave tardía) concentran los mayores retrasos de llegada en 2024? En la muestra se observa una alta concentración de vuelos en pocas aerolíneas principales (WN con 20,2 %, DL con 14,6 % y AA con 13,8 %) |